In [1]:
!pip install pyarrow fastparquet

In [2]:
import pandas as pd
import numpy as np

In [132]:
events = pd.read_parquet("C:/Users/treye/Downloads/events.parquet")
stints = pd.read_parquet("C:/Users/treye/Downloads/stints.parquet")
games = pd.read_parquet("C:/Users/treye/Downloads/games.parquet")
players = pd.read_parquet("C:/Users/treye/Downloads/players.parquet")
tracking = pd.read_parquet("C:/Users/treye/Downloads/tracking.parquet")

In [133]:
pd.set_option('display.max_columns', None)

In [134]:
events["dzone"] = (events["x_adj"] < -25.0).astype(int)
events["nzone"] = ((events["x_adj"] >= -25.0) & (events["x_adj"] <= 25.0)).astype(int)
events["ozone"] = (events["x_adj"] > 25.0).astype(int)
events["poss_team"] = events["team"]


g = events.groupby(["game_id", "sequence_id"])

events["prev_dzone"] = g["dzone"].shift(1)
events["prev_team"] = g["poss_team"].shift(1)
events["prev_nzone"] = g["nzone"].shift(1)
events["prev_ozone"] = g["ozone"].shift(1)
events["next_team"] = g["poss_team"].shift(-1)

events["lpr_poss_start"] = (
    (events["event_type"] == "lpr") &
    (events["poss_team"] != events["prev_team"])
).astype(int)

events["poss_id"] = (
    events
    .groupby("game_id")["lpr_poss_start"]
    .cumsum()
)

events["dz_poss_start"] = (
    (events["lpr_poss_start"] == 1) &
    (events["dzone"] == 1)
).astype(int)

In [138]:
events["dz_start_possession"] = events.groupby("poss_id")["dz_poss_start"].transform("max")

# 3 outcomes for and end (first and last are considered successes for getting puck out, second is a failure)
events["controlled_exit"] = ((events["dz_start_possession"]==1)&(events["dzone"]==0) & (events["prev_dzone"]==1) & (events["poss_team"]==events["prev_team"])).astype(int)
events["dz_turnover"] = ((events["dz_start_possession"]==1)&(events["event_type"]=="lpr") & (events["prev_dzone"]==1) & (events["ozone"]==1) & (events["poss_team"]!=events["prev_team"])).astype(int)
events["leave_dz_without_puck"] = ((events["dz_start_possession"]==1)&(events["ozone"]==0)&(events["prev_dzone"]==1)&(events["poss_team"]!=events["prev_team"])&(events["event_type"]=="lpr")).astype(int)

In [139]:
events.head(60)

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,prev_dzone,prev_team,prev_nzone,prev_ozone,next_team,lpr_poss_start,poss_id,dz_poss_start,dz_start_possession,controlled_exit,dz_turnover,leave_dz_without_puck
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,None,None,None,None,None,None,faceoff,None,None,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,NaN,NaN,NaN,NaN,GR,0,0,0,0,0,0,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,faceoff,failed,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1,0,1,0,GR,0.0,None,0.0,0.0,CLE,0,0,0,0,0,0,0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,faceoff,successful,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1,0,1,0,CLE,0.0,GR,1.0,0.0,CLE,0,0,0,0,0,0,0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,None,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1,0,1,0,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1,0,1,0,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.200000000,1.0,5,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,6.339600,20.917810,-6.339600,-20.917810,1,1,0,1,0,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.470000000,1.0,6,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS north+,north,NaN,6.339600,23.386793,-6.339600,-23.386793,1,1,0,1,0,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3.370000000,1.0,7,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,-23.839119,20.872547,23.839119,-20.872547,1,1,0,1,0,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,4.430000000,1.0,8,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,failed,eastwest,NORTH CYCLE OFFBOARDS-,northoffboards,NaN,-43.451580,25.992952,43.451580,-25.992952,1,1,0,0,1,CLE,0.0,CLE,1.0,0.0,CLE,0,0,0,0,0,0,0
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,7.130000000,1.0,9,1,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,failed,None,MISS PASS,regular,NaN,-92.236880,-20.369087,92.236880,20.369087,0,0,0,0,1,CLE,0.0,CLE,0.0,1.0,GR,0,0,0,0,0,0,0


In [146]:
events["dz_start_possession"].value_counts()

dz_start_possession
1    1796959
0       3505
Name: count, dtype: int64

In [142]:
events["controlled_exit"].value_counts()

controlled_exit
0    1721483
1      78981
Name: count, dtype: int64

In [144]:
events["dz_turnover"].value_counts()

dz_turnover
0    1740548
1      59916
Name: count, dtype: int64

In [145]:
events["leave_dz_without_puck"].value_counts()

leave_dz_without_puck
0    1774927
1      25537
Name: count, dtype: int64

In [103]:
events["dzone"] = (events["x_adj"] < -25.0).astype(int)
events["nzone"] = ((events["x_adj"] >= -25.0) & (events["x_adj"] <= 25.0)).astype(int)
events["ozone"] = (events["x_adj"] > 25.0).astype(int)
events["poss_team"] = events["team"]

g = events.groupby(["game_id", "sequence_id"])

events["prev_dzone"] = g["dzone"].shift(1)
events["prev_team"] = g["poss_team"].shift(1)
events["prev_nzone"] = g["nzone"].shift(1)
events["prev_ozone"] = g["ozone"].shift(1)
events["next_team"] = g["poss_team"].shift(-1)

# Start: team gains possession in dzone AND maintains it for next event
events["dz_possession_start"] = (
    (events["dzone"] == 1) &
    (events["event_type"]=="lpr") &
    (events["poss_team"] != events["prev_team"]) &
    (events["poss_team"] == events["next_team"])  # Keep this!
).astype(int)

events["poss_id"] = events.groupby("game_id")["dzpossession_start"].cumsum()
events["

events["dz_exit"] = (
    #team exits the zone with possession
    ((((events["nzone"] == 1) | (events["ozone"] == 1)) & 
    (events["prev_dzone"]==1) &
    (events["poss_team"] == events["prev_team"]))
    #team gets the puck out of the zone, loses possession in doing so
    | (((events["nzone"] == 1) | (events["dzone"] == 1)) & (events["event_type"]=="lpr") & 
       (events["prev_dzone"] == 1) &
        (events["poss_team"] != events["prev_team"])))
).astype(int)

events["dz_turnover"] = (
    ((events["ozone"] == 1) & (events["event_type"] == "lpr") &
    (events["prev_dzone"] == 1) &
    (events["poss_team"] != events["prev_team"]))
).astype(int)

In [122]:
events["next_event"]=events["event_type"].shift(-1)

In [126]:
events["event_type"].value_counts()

event_type
pass                      402670
lpr                       338836
reception                 304903
carry                     110362
failedpasslocation         97366
faceoff                    84621
block                      66233
puckprotection             54827
shot                       53971
pressure                   53961
controlledentryagainst     38081
dumpout                    35152
dumpin                     34740
dumpinagainst              32182
whistle                    25397
check                      24992
penalty                    11547
goal                        5667
controlledbreakout          4956
assist                      4793
icing                       4052
penaltydrawn                3765
receptionprevention         2353
deflection                  2047
offside                     2003
solpr                        237
socarry                      237
soshot                       231
sogoal                        76
shootout                      70

In [125]:
events[events["event_type"]=="puckprotection"]["next_event"].value_counts()

next_event
lpr                       33341
pass                      10366
check                      2495
carry                      2400
puckprotection             2059
pressure                   1416
whistle                     689
penalty                     446
dumpout                     409
penaltydrawn                310
dumpin                      231
block                       231
offside                     144
controlledentryagainst      135
dumpinagainst                50
shot                         48
controlledbreakout           20
reception                    19
failedpasslocation           14
assist                        4
Name: count, dtype: int64

In [106]:
events[events["event_type"]=="failedpasslocation"]["next_event"].value_counts()

next_event
lpr                       84124
block                      3738
whistle                    3563
pass                       2345
pressure                    840
offside                     417
dumpout                     371
carry                       343
penalty                     324
puckprotection              227
reception                   194
receptionprevention         181
penaltydrawn                176
check                       157
shot                        112
dumpin                       81
dumpinagainst                73
icing                        60
assist                       14
controlledbreakout           10
failedpasslocation            9
controlledentryagainst        6
goal                          1
Name: count, dtype: int64

In [107]:
events["potential_end"].value_counts()

potential_end
0    1635913
1     164551
Name: count, dtype: int64

In [108]:
events.head(60)

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts,next_event,cum_potential_ends,in_dz_possession
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,None,None,None,None,None,None,faceoff,None,None,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,0,faceoff,0.0,0.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,faceoff,failed,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1,0,1,0,GR,0,0,0,0,0,faceoff,0.0,0.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,faceoff,successful,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1,0,1,0,CLE,0,0,0,0,0,lpr,0.0,0.0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,None,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1,0,1,0,CLE,0,0,0,0,0,reception,0.0,0.0
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.200000000,1.0,5,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,6.339600,20.917810,-6.339600,-20.917810,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.470000000,1.0,6,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS north+,north,NaN,6.339600,23.386793,-6.339600,-23.386793,1,1,0,1,0,CLE,0,0,0,0,0,reception,0.0,0.0
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3.370000000,1.0,7,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,-23.839119,20.872547,23.839119,-20.872547,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,4.430000000,1.0,8,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,failed,eastwest,NORTH CYCLE OFFBOARDS-,northoffboards,NaN,-43.451580,25.992952,43.451580,-25.992952,1,1,0,0,1,CLE,0,0,0,0,0,reception,0.0,0.0
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,7.130000000,1.0,9,1,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,failed,None,MISS PASS,regular,NaN,-92.236880,-20.369087,92.236880,20.369087,0,0,0,0,1,CLE,0,0,0,0,0,lpr,0.0,0.0


In [114]:
events["dz_possession_start"].value_counts()

dz_possession_start
0    1674981
1     125483
Name: count, dtype: int64

In [115]:
events["dz_exit"].value_counts()

dz_exit
0    1800458
1          6
Name: count, dtype: int64

In [116]:
events["dz_turnover"].value_counts()

dz_turnover
0    1800464
Name: count, dtype: int64

In [112]:
events[events["dz_turnover"]==1]["event_type"].value_counts()

Series([], Name: count, dtype: int64)

In [69]:
events[(events["event_type"]=="penalty")]

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts
217,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,257.870000000,34.0,217,3,663df049-a045-0561-16e3-1db633a0723e,"Aston-Reese, Zachary",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,penalty,successful,None,TRIPPING PENALTY OZ,tripping,NaN,61.660324,-18.358490,61.660324,-18.358490,1,1,0,0,1,GR,0,0,1,1,5
230,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,273.030000000,35.0,230,3,None,None,None,None,None,None,penalty,None,None,Penalty Start,start,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,6
295,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,332.030000000,43.0,295,4,None,None,None,None,None,None,penalty,None,None,Penalty End,end,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,4
534,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,576.700000000,62.0,534,9,e9ef1506-4a97-1586-e7b4-a1dfbec02e4e,"Fix-Wolansky, Trey",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,penalty,successful,None,ROUGHING PENALTY NZ,roughing,NaN,18.910332,37.970920,-18.910332,-37.970920,1,0,0,1,0,CLE,0,0,0,0,5
535,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,577.000000000,62.0,535,9,4772aa22-208f-44a0-8e47-0029da368b48,"Kasper, Marco",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,penalty,successful,None,ROUGHING PENALTY NZ,roughing,NaN,19.916214,38.473860,19.916214,38.473860,1,1,0,1,0,GR,0,0,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1799873,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,648.030000000,344.0,3009,41,None,None,None,None,None,None,penalty,None,None,Penalty Start,start,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,None,0,0,0,0,4
1799974,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,768.030000000,358.0,3110,43,None,None,None,None,None,None,penalty,None,None,Penalty End,end,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,None,0,0,0,0,2
1800019,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,821.830000000,361.0,3155,44,b90aa353-979c-854a-6f54-5a714d009b15,"Hall, Curtis",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,penalty,successful,None,SLASHING PENALTY NZ,slashing,NaN,-22.327347,9.808823,-22.327347,9.808823,1,1,0,1,0,GR,0,0,0,0,1
1800024,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,824.030000000,361.0,3160,44,None,None,None,None,None,None,penalty,None,None,Penalty Start,start,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,2


In [119]:
events.head(60)

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts,next_event,cum_potential_ends,in_dz_possession,potential_exit,potential_turnover
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,None,None,None,None,None,None,faceoff,None,None,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,0,faceoff,0.0,0.0,0,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,faceoff,failed,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1,0,1,0,GR,0,0,0,0,0,faceoff,0.0,0.0,0,0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,faceoff,successful,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1,0,1,0,CLE,0,0,0,0,0,lpr,0.0,0.0,0,0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,None,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0,0,0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1,0,1,0,CLE,0,0,0,0,0,reception,0.0,0.0,0,0
5,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.200000000,1.0,5,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,6.339600,20.917810,-6.339600,-20.917810,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0,0,0
6,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.470000000,1.0,6,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS north+,north,NaN,6.339600,23.386793,-6.339600,-23.386793,1,1,0,1,0,CLE,0,0,0,0,0,reception,0.0,0.0,0,0
7,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,3.370000000,1.0,7,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,None,N-ZONE PASS RECEPTION,regular,NaN,-23.839119,20.872547,23.839119,-20.872547,1,1,0,1,0,CLE,0,0,0,0,0,pass,0.0,0.0,0,0
8,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,4.430000000,1.0,8,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,failed,eastwest,NORTH CYCLE OFFBOARDS-,northoffboards,NaN,-43.451580,25.992952,43.451580,-25.992952,1,1,0,0,1,CLE,0,0,0,0,0,reception,0.0,0.0,0,0
9,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,7.130000000,1.0,9,1,0f052c34-6cd2-f4fd-a861-d9051a8e86e4,"Angle, Tyler",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,failed,None,MISS PASS,regular,NaN,-92.236880,-20.369087,92.236880,20.369087,0,0,0,0,1,CLE,0,0,0,0,0,lpr,0.0,0.0,0,0


In [49]:
events[events["dz_turnover"]==1]["event_type"].shift(1).value_counts()

event_type
lpr                    53121
shot                   39496
pass                   27953
puckprotection         18692
block                  18486
faceoff                10095
reception               9898
check                   5655
failedpasslocation      3624
carry                   3232
deflection              1971
penaltydrawn            1193
penalty                  902
receptionprevention      653
assist                   422
pressure                 182
dumpin                    79
offside                   77
goal                       3
sopuckprotection           1
Name: count, dtype: int64

In [26]:
events[events["event_type"]=="goal"]

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts
293,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,331.500000000,43.0,293,4,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,goal,successful,None,GOAL,none,NaN,-69.603810,-21.876472,69.603810,21.876472,1,1,0,0,1,CLE,0,0,0,0,4
294,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,332.030000000,43.0,294,4,None,None,None,None,None,None,goal,None,None,Goal,none,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,4
431,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,463.400000000,56.0,431,7,2f0d296c-e9df-b8b7-f0e0-a135b8802329,"Soderblom, Elmer",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,goal,successful,None,GOAL,none,NaN,88.822660,4.779411,88.822660,4.779411,0,0,0,0,1,GR,0,0,0,0,2
432,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,464.030000000,56.0,432,7,None,None,None,None,None,None,goal,None,None,Goal,none,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,2
1384,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,195.530000000,167.0,1384,19,e9ef1506-4a97-1586-e7b4-a1dfbec02e4e,"Fix-Wolansky, Trey",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,goal,successful,None,GOAL,none,NaN,85.800280,4.075298,85.800280,4.075298,1,0,0,0,1,CLE,0,0,0,0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1798554,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,2,511.030000000,207.0,1688,26,None,None,None,None,None,None,goal,None,None,Goal,none,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,19
1800228,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,1000.270000000,374.0,3364,47,2f0d296c-e9df-b8b7-f0e0-a135b8802329,"Soderblom, Elmer",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,goal,successful,None,GOAL,none,NaN,63.172653,5.282352,63.172653,5.282352,1,1,0,0,1,GR,0,0,0,0,5
1800229,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,1001.030000000,374.0,3365,47,None,None,None,None,None,None,goal,None,None,Goal,none,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,5
1800304,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,1068.670000000,382.0,3440,49,a855a24c-10d7-480c-5f02-f40cacd514f9,"Luff, Matt",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,goal,successful,None,GOAL,none,NaN,48.587357,35.961765,48.587357,35.961765,1,1,0,0,1,GR,0,0,0,0,5


In [14]:
events["event_type"].value_counts()

event_type
pass                      402670
lpr                       338836
reception                 304903
carry                     110362
failedpasslocation         97366
faceoff                    84621
block                      66233
puckprotection             54827
shot                       53971
pressure                   53961
controlledentryagainst     38081
dumpout                    35152
dumpin                     34740
dumpinagainst              32182
whistle                    25397
check                      24992
penalty                    11547
goal                        5667
controlledbreakout          4956
assist                      4793
icing                       4052
penaltydrawn                3765
receptionprevention         2353
deflection                  2047
offside                     2003
solpr                        237
socarry                      237
soshot                       231
sogoal                        76
shootout                      70

In [15]:
events[(events["event_type"]=="check") & (events["ozone"]==1)]

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts
338,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,368.830000000,47.0,338,5,8bd8e696-4514-a5d5-792b-80b002761f46,"Hirose, Taro",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,check,successful,None,OZ BODY CHK+,body,NaN,73.734420,-39.479410,73.734420,-39.479410,1,1,0,0,1,GR,0,0,1,1,6
461,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,504.030000000,60.0,461,8,015f554a-21c0-99bc-0a31-a176810b40c6,"Shine, Dominik",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,check,successful,None,OZ STICK CHK+,stick,NaN,97.372650,12.323528,97.372650,12.323528,0,0,0,0,1,GR,0,0,1,1,2
1493,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,304.570000000,172.0,1493,22,4447229a-e693-b7a3-6bbe-f464ae9917c4,"Pyyhtia, Mikael",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,check,successful,None,OZ STICK CHK+,stick,NaN,96.363266,0.250332,96.363266,0.250332,1,1,0,0,1,CLE,0,0,1,1,3
1499,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,309.770000000,172.0,1499,22,f7d87866-fb2a-6a94-5317-d30145102fef,"Meyer, Carson",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,check,successful,None,OZ STICK CHK+,stick,NaN,71.216220,34.450333,71.216220,34.450333,1,1,0,0,1,CLE,0,0,1,1,3
1625,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,439.630000000,185.0,1625,23,daa52256-1dc6-fde2-afc0-7599fd3359eb,"Hanas, Cross",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,check,successful,None,OZ STICK CHK+,stick,NaN,-95.257320,-4.276138,95.257320,4.276138,1,0,0,0,1,GR,0,0,1,1,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1800057,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,843.570000000,362.0,3193,45,ad4b002b-5fc6-4cdc-1d16-192ced8c9b16,"Gaudet, Jake",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,check,successful,None,OZ BODY CHK+,body,NaN,-91.741110,31.428326,91.741110,-31.428326,0,0,0,0,1,CLE,0,0,1,1,1
1800217,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,995.930000000,374.0,3353,47,daa52256-1dc6-fde2-afc0-7599fd3359eb,"Hanas, Cross",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,check,successful,None,OZ BODY CHK+,body,NaN,83.793240,-37.970590,83.793240,-37.970590,1,0,0,0,1,GR,0,0,1,1,5
1800254,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,1016.800000000,377.0,3390,49,0753b094-9e2f-976d-85c8-d22a1d280e8d,"Jiricek, David",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,check,successful,None,OZ BODY CHK+,body,NaN,-72.621460,41.494118,72.621460,-41.494118,1,1,0,0,1,CLE,0,0,1,1,1
1800273,ff8386b6-5f9b-4734-00e6-bc8b4c0a4cca,3,1035.500000000,377.0,3409,49,6845c1eb-216d-4366-cda0-d3c220357faf,"Del Bel Belluz, Luca",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,check,successful,None,OZ STICK CHK+,stick,NaN,-64.071465,-4.776470,64.071465,4.776470,0,0,0,0,1,CLE,0,0,1,1,2


In [16]:
events.head(341)

,game_id,period,period_time,game_stint,sl_event_id,sequence_id,player_id,player_name,team,team_id,opp_team,opp_team_id,event_type,outcome,flags,description,detail,sl_xg_all_shots,x,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dzone,nzone,ozone,poss_team,dz_possession_start,dz_exit,dz_turnover,potential_end,cum_starts
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,0,1,None,None,None,None,None,None,faceoff,None,None,Face-Off,nz,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,None,0,0,0,0,0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,1,8cdcb61e-d733-bde6-a101-ef3140e48149,"L'Esperance, Joel",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,faceoff,failed,"centerfodot, righthandedopponent",NZ FACE OFF-,none,NaN,-0.201431,-0.755550,-0.201431,-0.755550,1,1,0,1,0,GR,0,0,0,0,0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,2,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,faceoff,successful,"centerfodot, righthandedopponent",NZ REC FACE OFF+ENTRY,recoveredwithentry,NaN,-0.201431,-0.755550,0.201431,0.755550,1,1,0,1,0,CLE,0,0,0,0,0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,3,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,None,F/OFF LPR+ NZ,faceoff,NaN,0.807243,1.257381,-0.807243,-1.257381,1,1,0,1,0,CLE,0,0,0,0,0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,4,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,"McKown, Hunter",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,eastside,NZPASS south+,south,NaN,0.304306,0.845894,-0.304306,-0.845894,1,1,0,1,0,CLE,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,366.970000000,46.0,336,5,90e5f527-993f-a1ac-44b5-9fb226fa6128,"Whelan, Alex",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,failed,None,LPR- DZ,none,NaN,79.266770,-28.414703,-79.266770,28.414703,1,0,1,0,0,CLE,1,0,0,0,6
337,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,368.670000000,47.0,337,5,d54b6c97-2380-f382-7937-732841fca283,"Pearson, Justin",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,alongboards,LPR FROM SC DZ,none,NaN,76.752075,-40.485290,-76.752075,40.485290,1,0,1,0,0,CLE,0,0,0,0,6
338,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,368.830000000,47.0,338,5,8bd8e696-4514-a5d5-792b-80b002761f46,"Hirose, Taro",GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,check,successful,None,OZ BODY CHK+,body,NaN,73.734420,-39.479410,73.734420,-39.479410,1,1,0,0,1,GR,0,0,1,1,6
339,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,368.830000000,47.0,339,5,d54b6c97-2380-f382-7937-732841fca283,"Pearson, Justin",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,puckprotection,failed,None,DZPUCKPROTECTION-,body,NaN,73.734420,-38.976470,-73.734420,38.976470,1,0,1,0,0,CLE,0,0,0,0,6
